In [1]:
import pandas as pd

In [8]:
def process_csv(path):
    df = pd.read_csv(path)
    # 去掉第一行的值
    df = df.drop(0)
    df = df.drop(columns=df.columns[0])
    df = df.reset_index(drop=True)
    return df

In [9]:
test_report_path = 'labeled_reports.csv'
test_df = process_csv(test_report_path)
test_df.head()

,No Finding,Enlarged Cardiomediastinum,Cardiomegaly,Lung Lesion,Lung Opacity,Edema,Consolidation,Pneumonia,Atelectasis,Pneumothorax,Pleural Effusion,Pleural Other,Fracture,Support Devices
0,NaN,NaN,1.0,NaN,NaN,NaN,0.0,NaN,NaN,0.0,0.0,NaN,NaN,NaN
1,1.0,0.0,NaN,NaN,NaN,0.0,NaN,NaN,NaN,0.0,0.0,NaN,NaN,-1.0
2,NaN,-1.0,-1.0,NaN,1.0,1.0,-1.0,NaN,1.0,0.0,1.0,NaN,NaN,0.0
3,NaN,-1.0,-1.0,NaN,1.0,1.0,-1.0,NaN,1.0,0.0,1.0,NaN,NaN,0.0
4,NaN,NaN,1.0,NaN,NaN,-1.0,0.0,1.0,1.0,NaN,1.0,NaN,NaN,1.0


In [1]:
import pandas as pd
from sklearn.metrics import precision_score, recall_score, f1_score

def compute_metrics(pred_path, label_path) -> dict:
    """
    计算chexpert-labeler输出的宏平均 Precision、Recall 和 F1 分数。

    参数:
    - pred_df: pandas.DataFrame，包含预测的标签。
    - label_df: pandas.DataFrame，包含真实的标签。

    返回:
    - metrics: dict，包含 'macro_precision'、'macro_recall' 和 'macro_f1'。
    """
    
    # 读取CSV文件到DataFrame，并填充缺失值为 -1
    pred_df = pd.read_csv(pred_path).fillna(-1)
    label_df = pd.read_csv(label_path).fillna(-1)

    # 去掉第一行
    pred_df = pred_df.drop(0).reset_index(drop=True)
    label_df = label_df.drop(0).reset_index(drop=True)

    # 去掉第一列
    pred_df = pred_df.drop(columns=pred_df.columns[0])
    label_df = label_df.drop(columns=label_df.columns[0])

    # 确保预测和标签的列一致
    if list(pred_df.columns) != list(label_df.columns):
        raise ValueError("预测和标签的列不一致。请确保两者具有相同的标签列。")
    
    label_columns = label_df.columns.tolist()
    
    # 初始化列表来存储每个标签的指标
    precision_scores = []
    recall_scores = []
    f1_scores = []
    
    for label in label_columns:
        y_pred = pred_df[label]
        y_true = label_df[label]
        
        # 过滤掉真实标签为 -1 的样本（不确定）
        mask = y_true != -1
        y_pred_filtered = y_pred[mask]
        y_true_filtered = y_true[mask]
        
        # 如果过滤后没有样本，则跳过该标签
        if len(y_true_filtered) == 0:
            print(f"标签 '{label}' 中没有有效样本（所有样本均为不确定）。")
            continue
        
        # 将预测值和真实值中的 -1 替换为 0（否定）
        y_pred_binary = y_pred_filtered.replace(-1, 0)
        y_true_binary = y_true_filtered.replace(-1, 0)
        
        # 计算 Precision、Recall 和 F1 分数，忽略零除错误
        try:
            precision = precision_score(y_true_binary, y_pred_binary, average='binary', zero_division=0)
            recall = recall_score(y_true_binary, y_pred_binary, average='binary', zero_division=0)
            f1 = f1_score(y_true_binary, y_pred_binary, average='binary', zero_division=0)
            
            precision_scores.append(precision)
            recall_scores.append(recall)
            f1_scores.append(f1)
        except ValueError as e:
            print(f"标签 '{label}' 的指标计算失败: {e}")
    
    # 计算宏平均指标
    if not f1_scores:
        print("没有有效的标签用于计算宏平均指标。")
        return {"macro_precision": 0.0, "macro_recall": 0.0, "macro_f1": 0.0}
    
    macro_precision = round(sum(precision_scores) / len(precision_scores), 4)
    macro_recall = round(sum(recall_scores) / len(recall_scores), 4)
    macro_f1 = round(sum(f1_scores) / len(f1_scores), 4)
    
    metrics = {
        "macro_precision": macro_precision,
        "macro_recall": macro_recall,
        "macro_f1": macro_f1
    }
    
    return metrics


In [2]:
mimic_csv = 'mimic_label.csv'
CARZero_csv = 'CARZero_label.csv'
KAD_cvs = 'KAD.csv'
medclip_csv = 'medclip.csv'
CARZero_no_LLM = 'CARZero_no_LLM.csv'

CARZero_metrics = compute_metrics(mimic_csv, CARZero_csv)
KAD_metrics = compute_metrics(mimic_csv, KAD_cvs)
medclip_metrics = compute_metrics(mimic_csv, medclip_csv)
CARZero_no_LLM_metrics = compute_metrics(mimic_csv, CARZero_no_LLM)

print("CARZero:", CARZero_metrics)
print("KAD:", KAD_metrics)
print("medclip:", medclip_metrics)
print("CARZero_no_LLM:", CARZero_no_LLM_metrics)

标签 'Pleural Other' 中没有有效样本（所有样本均为不确定）。
CARZero: {'macro_precision': 0.6522, 'macro_recall': 0.1808, 'macro_f1': 0.2529}
KAD: {'macro_precision': 0.6893, 'macro_recall': 0.2051, 'macro_f1': 0.2823}
medclip: {'macro_precision': 0.4854, 'macro_recall': 0.1635, 'macro_f1': 0.216}
CARZero_no_LLM: {'macro_precision': 0.6699, 'macro_recall': 0.1812, 'macro_f1': 0.256}


In [7]:
VECL_v1_csv = 'VECL_v1.csv'
VECL_v2_csv = 'VECL_v2.csv'
soft_lable_plus_csv = 'soft-lable-plus.csv'

VECL_v1_metrics = compute_metrics(mimic_csv, VECL_v1_csv)
VECL_v2_metrics = compute_metrics(mimic_csv, VECL_v2_csv)
soft_lable_plus_metrics = compute_metrics(mimic_csv, soft_lable_plus_csv)


print("VECL_v1_csv:", VECL_v1_metrics)
print("VECL_v2_csv:", VECL_v2_metrics)
print("soft-lable-plus_csv:", soft_lable_plus_metrics)

VECL_v1_csv: {'macro_precision': 0.7, 'macro_recall': 0.2157, 'macro_f1': 0.2682}
VECL_v2_csv: {'macro_precision': 0.682, 'macro_recall': 0.188, 'macro_f1': 0.2675}
soft-lable-plus_csv: {'macro_precision': 0.4781, 'macro_recall': 0.2142, 'macro_f1': 0.227}
